# Worked Build: The Compliance-Checker Agent
**Module 5, Topic 3 — Automation & Agentic Systems**

## The Problem

A compliance officer asks: *"Is Sahel Link Networks compliant with Licence Condition 7.2?"* Answering requires searching three MCE documents, finding the relevant paragraphs, drafting a cited response, and presenting it for human review. A single LLM call cannot do this — it needs a **multi-step workflow** with retrieval, generation, and a review gate.

## The Solution

Build a **LangGraph agent** with five nodes — decide, search, draft, review, finalise — connected by conditional edges. The agent searches the MCE corpus, drafts a cited compliance response, and presents it for human approval before returning the answer.

In this notebook you will:
1. Load the synthetic MCE corpus
2. Build a keyword retrieval tool
3. Define a 5-node graph with two conditional edges
4. Test against compliance, enforcement, incident, and unanswerable questions
5. Verify the agent cites documents and refuses when evidence is missing

**Time:** 40 minutes | **Difficulty:** ⭐⭐⭐⭐☆

## Setup

In [ ]:
!pip install langgraph openai python-dotenv -q

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

print(f"Client ready — model: {MODEL}")

Client ready — model: google/gemma-4-E26B-it


## Step 1 — Load the MCE Corpus

Three synthetic documents from `shared/synthetic-mce-corpus/`.  
We load them as strings so our search function can work over them.

In [12]:
import os

CORPUS_DIR = os.path.join("..", "..", "..", "..", "shared", "synthetic-mce-corpus")


def load_doc(filename):
    path = os.path.join(CORPUS_DIR, filename)
    with open(path, encoding="utf-8") as f:
        return f.read()

CORPUS = {
    "MCE-CN-2026-031": load_doc("MCE-CN-2026-031_compliance_notice.txt"),
    "MCE-EC-2025-118": load_doc("MCE-EC-2025-118_enforcement_case_file.txt"),
    "NOC-INC-2026-0417": load_doc("NOC-INC-2026-0417_noc_incident_report.txt"),
}

for doc_id, text in CORPUS.items():
    print(f"{doc_id}: {len(text)} chars")

MCE-CN-2026-031: 3021 chars
MCE-EC-2025-118: 2673 chars
NOC-INC-2026-0417: 3421 chars


## Step 2 — The Retrieval Tool

A simple keyword search over the corpus. Each matching paragraph is returned with its document ID for citation.

In [13]:
def search_corpus(query: str) -> list[dict]:
    """Search the MCE corpus by keyword and return matching excerpts."""
    terms = [t.lower() for t in query.split() if len(t) > 2]
    results = []
    for doc_id, text in CORPUS.items():
        for para in text.split("\n\n"):
            if any(term in para.lower() for term in terms):
                results.append({"doc_id": doc_id, "excerpt": para.strip()[:500]})
    return results


# Quick test
test_results = search_corpus("Sahel Link compliance")
print(f"Found {len(test_results)} results for 'Sahel Link compliance'")
for r in test_results[:2]:
    print(f"  [{r['doc_id']}] {r['excerpt'][:80]}...")

Found 13 results for 'Sahel Link compliance'
  [MCE-CN-2026-031] MONITORING, COMPLIANCE AND ENFORCEMENT (MCE)
NOTICE OF NON-COMPLIANCE...
  [MCE-CN-2026-031] Notice reference:       MCE-CN-2026-031
Date of notice:         2026-03-09
Issue...


## Step 3 — Define the State

`search_results` uses `Annotated[list, add]` so multiple searches append instead of replacing.

In [14]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END


class ComplianceState(TypedDict):
    question: str
    search_results: Annotated[list, add]
    draft: str
    needs_search: bool
    approved: bool
    answer: str


print("State schema:", list(ComplianceState.__annotations__.keys()))

State schema: ['question', 'search_results', 'draft', 'needs_search', 'approved', 'answer']


## Step 4 — Write the Nodes

Five nodes, each a plain function:
- `decide` — asks the LLM whether a document search is needed
- `search` — runs the retrieval tool
- `draft` — writes a cited compliance response
- `review` — presents for human approval (auto-approve in demo)
- `finalise` — promotes the approved draft to the answer

In [15]:
def decide(state: ComplianceState) -> dict:
    """Ask the LLM whether a document search is needed."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a compliance assistant. Decide if the following "
                    "question requires searching compliance documents to answer. "
                    "Reply with exactly YES or NO."
                ),
            },
            {"role": "user", "content": state["question"]},
        ],
    )
    answer = response.choices[0].message.content.strip().upper()
    needs = answer.startswith("YES")
    print(f"[decide] needs_search = {needs}")
    return {"needs_search": needs}


print("decide node defined")

decide node defined


In [16]:
def search(state: ComplianceState) -> dict:
    """Run the retrieval tool and add results to state."""
    results = search_corpus(state["question"])
    print(f"[search] found {len(results)} results")
    return {"search_results": results}


print("search node defined")

search node defined


In [17]:
def draft(state: ComplianceState) -> dict:
    """Draft a compliance response citing the documents found."""
    if state["search_results"]:
        context = "\n\n".join(
            f"[{r['doc_id']}]: {r['excerpt']}"
            for r in state["search_results"]
        )
    else:
        context = "(no documents found)"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a compliance assistant. Answer the question using "
                    "only the documents provided below. Cite each fact with the "
                    "document reference in square brackets, e.g. [MCE-CN-2026-031]. "
                    "If the documents do not contain the answer, say so explicitly."
                ),
            },
            {
                "role": "user",
                "content": f"Question: {state['question']}\n\nDocuments:\n{context}",
            },
        ],
    )
    draft_text = response.choices[0].message.content
    print(f"[draft] wrote {len(draft_text)} chars")
    return {"draft": draft_text}


print("draft node defined")

draft node defined


In [18]:
def review(state: ComplianceState) -> dict:
    """Present the draft for human review."""
    print("=" * 60)
    print("DRAFT FOR REVIEW")
    print("=" * 60)
    print(state["draft"])
    print("=" * 60)
    print("[auto-approved for demo]")
    return {"approved": True}


def finalise(state: ComplianceState) -> dict:
    """Promote the approved draft to the final answer."""
    print("[finalise] draft approved and promoted to answer")
    return {"answer": state["draft"]}


print("review and finalise nodes defined")

review and finalise nodes defined


## Step 5 — Routing Functions and Graph Wiring

Two decision points:  
1. After decide — search or skip to draft  
2. After review — finalise or loop back to search

In [19]:
def route_after_decide(state: ComplianceState) -> str:
    if state["needs_search"]:
        return "search"
    return "draft"


def route_after_review(state: ComplianceState) -> str:
    if state["approved"]:
        return "finalise"
    return "search"


# Build the graph
graph = StateGraph(ComplianceState)

graph.add_node("decide", decide)
graph.add_node("search", search)
graph.add_node("draft", draft)
graph.add_node("review", review)
graph.add_node("finalise", finalise)

graph.add_edge(START, "decide")
graph.add_conditional_edges(
    "decide",
    route_after_decide,
    {"search": "search", "draft": "draft"},
)
graph.add_edge("search", "draft")
graph.add_edge("draft", "review")
graph.add_conditional_edges(
    "review",
    route_after_review,
    {"finalise": "finalise", "search": "search"},
)
graph.add_edge("finalise", END)

app = graph.compile()

print("Graph compiled successfully")
print()
print("Graph structure:")
print("START -> decide -> [needs_search?]")
print("  YES -> search -> draft -> review -> [approved?]")
print("  NO  -> draft ---^                    YES -> finalise -> END")
print("                                       NO  -> search (loop)")

Graph compiled successfully

Graph structure:
START -> decide -> [needs_search?]
  YES -> search -> draft -> review -> [approved?]
  NO  -> draft ---^                    YES -> finalise -> END
                                       NO  -> search (loop)


## Step 6 — Test: Compliance Question (Needs Search)

Expected: the agent searches, finds MCE-CN-2026-031, and drafts a cited response about the missing Q4 2025 report.

In [20]:
result1 = app.invoke({
    "question": "Is Sahel Link Networks currently compliant with Licence Condition 7.2?",
    "search_results": [],
    "draft": "",
    "needs_search": False,
    "approved": False,
    "answer": "",
})

print("\nFINAL ANSWER:")
print(result1["answer"])

[decide] needs_search = True
[search] found 15 results
[draft] wrote 512 chars
DRAFT FOR REVIEW
No, Sahel Link Networks Ltd is not currently compliant with Licence Condition 7.2 (Quarterly Reporting) [MCE-CN-2026-031]. 

The company failed to submit its Quarterly Service Quality Report for the fourth quarter of 2025 (1 October 2025 to 31 December 2025), which was due on 30 January 2026 [MCE-CN-2026-031]. While a revision notice issued on 2026-03-20 extended the deadline to submit the report to 2026-04-13, the failure to submit the report constitutes a breach of Licence Condition 7.2 [MCE-CN-2026-031].
[auto-approved for demo]
[finalise] draft approved and promoted to answer

FINAL ANSWER:
No, Sahel Link Networks Ltd is not currently compliant with Licence Condition 7.2 (Quarterly Reporting) [MCE-CN-2026-031]. 

The company failed to submit its Quarterly Service Quality Report for the fourth quarter of 2025 (1 October 2025 to 31 December 2025), which was due on 30 January 2026 [MCE-CN-2

### Check the response

- Does it cite **MCE-CN-2026-031**?
- Does it mention the **Q4 2025** report is missing?
- Does it give the **revised deadline 2026-04-13** (not the original 2026-03-30)?
- Does it mention the **penalty of NGN 2,500,000 per week**?

## Step 7 — Test: Enforcement Question

In [ ]:
result2 = app.invoke({
    "question": "What was the outcome of the Delta Coast Broadband enforcement case?",
    "search_results": [],
    "draft": "",
    "needs_search": False,
    "approved": False,
    "answer": "",
})

print("\nFINAL ANSWER:")
print(result2["answer"])

### Check: cites **MCE-EC-2025-118**, says case is **closed**, mentions **NGN 5,000,000 penalty paid in full**.

## Step 8 — Test: Question the Corpus Cannot Answer

In [ ]:
result3 = app.invoke({
    "question": "Who is the Director of Monitoring, Compliance and Enforcement?",
    "search_results": [],
    "draft": "",
    "needs_search": False,
    "approved": False,
    "answer": "",
})

print("\nFINAL ANSWER:")
print(result3["answer"])

### Check: says documents **do not contain** this information and **avoids making up a name**.

## Step 9 — Test: NOC Incident Question

The trap: the summary says "approximately two hours" but the timestamps give 1 hour 38 minutes.

In [ ]:
result4 = app.invoke({
    "question": "What caused the NOC incident on 17 April 2026 and how long did it last?",
    "search_results": [],
    "draft": "",
    "needs_search": False,
    "approved": False,
    "answer": "",
})

print("\nFINAL ANSWER:")
print(result4["answer"])

### Check: cites **NOC-INC-2026-0417**, root cause is **failed cooling fan on HPA-2**, duration from **timestamps (1h 38m)** vs narrative's **"approximately two hours"**.

## Summary

You built a compliance-checker agent with:
- **5 nodes:** decide, search, draft, review, finalise
- **2 conditional edges:** search-or-skip after decide, finalise-or-loop after review
- **Retrieval tool:** keyword search over the MCE corpus
- **Cited responses:** the system prompt forces citations and refuses when documents do not contain the answer
- **Human-in-the-loop:** the review node (auto-approve in demo, human approval in production)

In Topic 4 we replace the hand-written retrieval with LangChain components.

## Exercises

### Exercise 1 — Implement real human review
Modify the `review` node to use `input()` to ask the user to type `yes` or `no`. If they type `no`, set `approved` to `False` so the graph loops back to search. Test with a question where the first draft misses a fact — reject it and verify the agent searches again.

### Exercise 2 — Add a `confidence` field to state
Add a `confidence: str` field to `ComplianceState`. Modify the `draft` node's system prompt to also output a confidence level (`high`, `medium`, or `low`) before the answer. Parse it into the state. Modify `route_after_review` so that `low` confidence drafts are automatically sent back for another search round.

### Exercise 3 — Cross-document question
Ask the agent: *"Was the NOC incident on 17 April related to Sahel Link Networks' compliance case?"* The answer requires evidence from both NOC-INC-2026-0417 and MCE-CN-2026-031. Check whether the keyword search finds both documents. If not, how would you improve the search?

**Time:** 25 minutes

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here
